In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [10]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [15]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.AdaHessian(model=model, lr_init=1e-4)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5075364708900452
epoch:  1, loss: 0.5055214166641235
epoch:  2, loss: 0.5052785277366638
epoch:  3, loss: 0.505077064037323
epoch:  4, loss: 0.5045876502990723
epoch:  5, loss: 0.5046434998512268
epoch:  6, loss: 0.5044944286346436
epoch:  7, loss: 0.5043988823890686
epoch:  8, loss: 0.5043383836746216
epoch:  9, loss: 0.5046713352203369
epoch:  10, loss: 0.5044206380844116
epoch:  11, loss: 0.5044135451316833
epoch:  12, loss: 0.49992620944976807
epoch:  13, loss: 0.49961838126182556
epoch:  14, loss: 0.4994325339794159
epoch:  15, loss: 0.49932861328125
epoch:  16, loss: 0.49927589297294617
epoch:  17, loss: 0.4991300702095032
epoch:  18, loss: 0.4988801181316376
epoch:  19, loss: 0.4988919198513031
epoch:  20, loss: 0.49856680631637573
epoch:  21, loss: 0.4983969032764435
epoch:  22, loss: 0.4984361529350281
epoch:  23, loss: 0.49783802032470703
epoch:  24, loss: 0.49774500727653503
epoch:  25, loss: 0.4975937604904175
epoch:  26, loss: 0.49746865034103394
epoch: 

In [17]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -21042.113097676
Test metrics:  R2 = -27650.787318214403


In [18]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.AdaHessianLS(model=model, lr_init=1)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.27195632457733154
epoch:  1, loss: 0.047064442187547684
epoch:  2, loss: 0.04191133379936218
epoch:  3, loss: 0.041669998317956924
epoch:  4, loss: 0.040643759071826935
epoch:  5, loss: 0.040643759071826935
epoch:  6, loss: 0.040511250495910645
epoch:  7, loss: 0.040511250495910645
epoch:  8, loss: 0.03910096734762192
epoch:  9, loss: 0.03910096734762192
epoch:  10, loss: 0.03910096734762192
epoch:  11, loss: 0.038577817380428314
epoch:  12, loss: 0.038577817380428314
epoch:  13, loss: 0.037803757935762405
epoch:  14, loss: 0.03774299845099449
epoch:  15, loss: 0.03774299845099449
epoch:  16, loss: 0.0376308336853981
epoch:  17, loss: 0.0376308336853981
epoch:  18, loss: 0.037606529891490936
epoch:  19, loss: 0.03748070076107979
epoch:  20, loss: 0.03748070076107979
epoch:  21, loss: 0.03748070076107979
epoch:  22, loss: 0.03673514351248741
epoch:  23, loss: 0.03673514351248741
epoch:  24, loss: 0.03667433187365532
epoch:  25, loss: 0.03605442866683006
epoch:  26, lo

In [19]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.649539473592504
Test metrics:  R2 = 0.6515506893629368
